<a href="https://colab.research.google.com/github/VitorBZS/PLN-A940-M-D.S.M.-297-20262/blob/main/Aula06pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##0. Setup do ambiente

In [10]:
!pip install -q nltk spacy scikit-learn pandas
!python -m spacy download pt_core_news_md -q

import nltk
import spacy
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords', quiet=True)

nlp = spacy.load("pt_core_news_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 25.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


##1. Corpus

In [17]:
corpus = [

    "Quero saber o valor do condomínio do apartamento no centro.",

    "Aceita financiamento pela Caixa Econômica?",

    "Ainda está aceito para locação? Tenho interesse!",

    "Qual o valor do IPTU e do aluguel na região?"

]

for i, doc in enumerate(corpus, 1):
  print(f"Documento {i}: {doc}")


Documento 1: Quero saber o valor do condomínio do apartamento no centro.
Documento 2: Aceita financiamento pela Caixa Econômica?
Documento 3: Ainda está aceito para locação? Tenho interesse!
Documento 4: Qual o valor do IPTU e do aluguel na região?


##2. Bag of Words

In [18]:
from nltk.corpus import stopwords
stop_pt = list(stopwords.words('portuguese'))

vectorizer_bow = CountVectorizer(stop_words=stop_pt)
matriz_bow = vectorizer_bow.fit_transform(corpus)

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer_bow.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)
df_bow

,aceita,aceito,ainda,aluguel,apartamento,caixa,centro,condomínio,econômica,financiamento,interesse,iptu,locação,quero,região,saber,valor
Doc 1,0,0,0,0,1,0,1,1,0,0,0,0,0,1,0,1,1
Doc 2,1,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0
Doc 3,0,1,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0
Doc 4,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1


##2.2. Lematizando antes de vetorizar

In [ ]:
def lematizar(texto)
  doc = nlp(texto)
  return " ".join

##3. TF-IDF

In [13]:
vectorizer_tfidf = TfidfVectorizer(stop_words=stop_pt)
matriz_tfidf = vectorizer_tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),
    columns=vectorizer_tfidf.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
).round(2)
df_tfidf

,aceita,ainda,aluguel,apartamento,caixa,centro,condomínio,disponível,econômica,financiamento,interesse,iptu,locação,quero,região,saber,valor
Doc 1,0.0,0.0,0.00,0.42,0.0,0.42,0.42,0.0,0.0,0.0,0.0,0.00,0.0,0.42,0.00,0.42,0.33
Doc 2,0.5,0.0,0.00,0.00,0.5,0.00,0.00,0.0,0.5,0.5,0.0,0.00,0.0,0.00,0.00,0.00,0.00
Doc 3,0.0,0.5,0.00,0.00,0.0,0.00,0.00,0.5,0.0,0.0,0.5,0.00,0.5,0.00,0.00,0.00,0.00
Doc 4,0.0,0.0,0.53,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.53,0.0,0.00,0.53,0.00,0.41


##4. Leads Parecidos

In [14]:
similiariedade = cosine_similarity(matriz_tfidf)

df_tfidf = pd.DataFrame(
    similiariedade.round(2),
    columns=[f"Doc {i+1}" for i in range(len(corpus))],
    index = [f"Doc {i+1}" for i in range(len(corpus))]
)
df_tfidf

,Doc 1,Doc 2,Doc 3,Doc 4
Doc 1,1.00,0.0,0.0,0.14
Doc 2,0.00,1.0,0.0,0.00
Doc 3,0.00,0.0,1.0,0.00
Doc 4,0.14,0.0,0.0,1.00


##5. Word Embeddings

In [15]:
palavra1 = nlp("apartamento")

palavra2 = nlp("imóvel")

palavra3 = nlp("gato")

print("Similaridade apartamento x imóvel:", palavra1.similarity(palavra2))
print("Similaridade apartamento x gato:  ", palavra1.similarity(palavra3))


Similaridade apartamento x imóvel: 0.666731595993042
Similaridade apartamento x gato:   0.18601803481578827


##6. Embeddings - Alem da contagem

In [16]:
banco_1 = nlp("O banco aprovou o empréstimo.")
banco_2 = nlp("Sentei no banco da praça.")

# Buscar o token "banco" em cada frase pelo texto, em vez de pela posição
# (mais seguro do que "adivinhar" a posição, que muda conforme a tokenização)
token_banco_1 = [t for t in banco_1 if t.text.lower() == "banco"][0]
token_banco_2 = [t for t in banco_2 if t.text.lower() == "banco"][0]

vetor_banco_1 = token_banco_1.vector
vetor_banco_2 = token_banco_2.vector

import numpy as np
print("Os vetores são idênticos?", np.array_equal(vetor_banco_1, vetor_banco_2))

Os vetores são idênticos? True
